# 20. Time Series Forecasting: SSA + Combinatorial Models (RNN, LSTM, BPNN) + GWO

Implements the **combinatorial model** from the paper:
**"Time Series Forecasting Based on Combinatorial Models and Optimization"** (ICAACE 2024).

## Paper (adapted to our setting)
- **SSA:** Decompose time series (trajectory matrix → SVD → diagonal averaging); select first **10** components for reconstruction (denoising).
- **Three predictors:** **RNN**, **LSTM**, and **BPNN** (MLP) each predict the next 24h from the (optionally SSA-reconstructed) 72h input.
- **GWO (Gray Wolf Optimization):** Optimize **combination weights** w_rnn, w_lstm, w_bpnn (sum=1) so that **y_pred = w_rnn·y_rnn + w_lstm·y_lstm + w_bpnn·y_bpnn** minimizes validation MSE.

## Same setup as 10–19
Data: work_dir/final, 72h→24h. Metrics: MAE, RMSE, MASE. Models: SSA+RNN/LSTM/BPNN+GWO, Naive. Same visuals.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
print(f'TensorFlow: {tf.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'):
            _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU. On Apple Silicon: pip install tensorflow-metal')
USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists():
        return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try:
            dfs.append(pd.read_parquet(p))
        except Exception as e:
            print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df, test_df, lookback=72, forecast_horizon=24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    X_tr, y_tr, X_te, y_te = [], [], [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_tr.append(tr[i:i+lookback])
            y_tr.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                inp = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                h = max(0, lookback - d * forecast_horizon)
                inp = np.concatenate([tr[-h:], te[0:d*forecast_horizon]]) if h > 0 else te[d*forecast_horizon - lookback:d*forecast_horizon]
            tgt = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(inp) == lookback and len(tgt) == forecast_horizon:
                X_te.append(inp)
                y_te.append(tgt)
    if not X_tr or not X_te:
        return np.array([]), np.array([]), np.array([]), np.array([])
    return np.array(X_tr).reshape(-1, lookback, 1), np.array(y_tr), np.array(X_te).reshape(-1, lookback, 1), np.array(y_te)

In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
for band in class_options:
    if band not in train_data_by_band:
        continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(train_data_by_band[band], test_data_by_band[band], LOOKBACK, FORECAST_HORIZON)
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## SSA reconstruction (first 10 components, paper)
Optional: replace each 72h input by SSA-reconstructed series (denoising). We use raw 72h and train three models; paper uses reconstructed series.

In [ ]:
L_SSA = 20
N_COMP_RECON = 10

def _diag_average(X):
    L, K = X.shape
    n = L + K - 1
    y = np.zeros(n)
    for k in range(n):
        i_lo = max(0, k - K + 1)
        i_hi = min(k, L - 1)
        y[k] = np.mean([X[i, k - i] for i in range(i_lo, i_hi + 1)])
    return y

def ssa_reconstruct(series: np.ndarray, L: int, n_comp: int):
    x = series.astype(float)
    n = len(x)
    if L < 2 or n < L:
        return x.copy()
    K = n - L + 1
    X = np.column_stack([x[i:i+L] for i in range(K)])
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    m = min(n_comp, len(s))
    out = np.zeros(n)
    for i in range(m):
        Xi = (U[:, [i]] * s[i]) @ Vt[[i], :]
        out += _diag_average(Xi)
    return out

def ssa_reconstruct_batch(X_batch, L, n_comp):
    out = np.zeros_like(X_batch)
    for i in range(X_batch.shape[0]):
        out[i, :, 0] = ssa_reconstruct(X_batch[i, :, 0], L, n_comp)
    return out

X_train_ssa = ssa_reconstruct_batch(X_train, L_SSA, N_COMP_RECON)
X_test_ssa = ssa_reconstruct_batch(X_test, L_SSA, N_COMP_RECON)
print(f'SSA reconstruction done (first {N_COMP_RECON} components).')

## Train RNN, LSTM, BPNN (paper: three combinatorial models)

In [ ]:
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_tr_flat = X_train_ssa.reshape(-1, 1)
X_te_flat = X_test_ssa.reshape(-1, 1)
y_tr_flat = y_train.reshape(-1, 1)
X_train_s = scaler_X.fit_transform(X_tr_flat).reshape(X_train_ssa.shape)
X_test_s = scaler_X.transform(X_te_flat).reshape(X_test_ssa.shape)
y_train_s = scaler_y.fit_transform(y_tr_flat).reshape(y_train.shape)

BATCH = 128 if USE_GPU else 32
EPOCHS = 50

def build_rnn():
    inp = layers.Input(shape=(LOOKBACK, 1))
    x = layers.SimpleRNN(32, activation='tanh')(inp)
    out = layers.Dense(FORECAST_HORIZON, activation='linear')(x)
    return keras.Model(inp, out)

def build_lstm():
    inp = layers.Input(shape=(LOOKBACK, 1))
    x = layers.LSTM(32, activation='tanh')(inp)
    out = layers.Dense(FORECAST_HORIZON, activation='linear')(x)
    return keras.Model(inp, out)

def build_bpnn():
    inp = layers.Input(shape=(LOOKBACK, 1))
    x = layers.Flatten()(inp)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(FORECAST_HORIZON, activation='linear')(x)
    return keras.Model(inp, out)

model_rnn = build_rnn()
model_lstm = build_lstm()
model_bpnn = build_bpnn()
for m in [model_rnn, model_lstm, model_bpnn]:
    m.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])

model_rnn.fit(X_train_s, y_train_s, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)
model_lstm.fit(X_train_s, y_train_s, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)
model_bpnn.fit(X_train_s, y_train_s, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)
print('RNN, LSTM, BPNN trained.')

## GWO: optimize combination weights (w_rnn, w_lstm, w_bpnn)

In [ ]:
n_val = max(1, int(0.2 * len(X_train_s)))
X_val = X_train_s[-n_val:]
y_val = y_train_s[-n_val:]

pred_rnn_val = model_rnn.predict(X_val, verbose=0)
pred_lstm_val = model_lstm.predict(X_val, verbose=0)
pred_bpnn_val = model_bpnn.predict(X_val, verbose=0)

def weights_to_simplex(w3):
    w = np.maximum(w3, 0)
    s = w.sum()
    return w / s if s > 1e-9 else np.array([1/3, 1/3, 1/3])

def mse_combined(w):
    w = weights_to_simplex(w)
    y_comb = w[0]*pred_rnn_val + w[1]*pred_lstm_val + w[2]*pred_bpnn_val
    return np.mean((y_val - y_comb) ** 2)

n_pop = 30
n_iter = 100
lb = np.zeros(3)
ub = np.ones(3)
np.random.seed(42)
pop = np.random.uniform(lb, ub, (n_pop, 3))
pop = np.array([weights_to_simplex(p) for p in pop])

fitness = np.array([mse_combined(p) for p in pop])
idx = np.argsort(fitness)
alpha_pos = pop[idx[0]].copy()
beta_pos = pop[idx[1]].copy()
delta_pos = pop[idx[2]].copy()

for it in range(n_iter - 1):
    a = 2.0 - 2.0 * (it + 1) / n_iter
    for i in range(n_pop):
        r1, r2 = np.random.rand(3), np.random.rand(3)
        A1, C1 = 2*a*r1 - a, 2*r2
        D_alpha = np.abs(C1 * alpha_pos - pop[i])
        X1 = alpha_pos - A1 * D_alpha
        r1, r2 = np.random.rand(3), np.random.rand(3)
        A2, C2 = 2*a*r1 - a, 2*r2
        D_beta = np.abs(C2 * beta_pos - pop[i])
        X2 = beta_pos - A2 * D_beta
        r1, r2 = np.random.rand(3), np.random.rand(3)
        A3, C3 = 2*a*r1 - a, 2*r2
        D_delta = np.abs(C3 * delta_pos - pop[i])
        X3 = delta_pos - A3 * D_delta
        pop[i] = (X1 + X2 + X3) / 3.0
        pop[i] = np.clip(pop[i], 0, 1)
        pop[i] = weights_to_simplex(pop[i])
    fitness = np.array([mse_combined(p) for p in pop])
    idx = np.argsort(fitness)
    alpha_pos = pop[idx[0]].copy()
    beta_pos = pop[idx[1]].copy()
    delta_pos = pop[idx[2]].copy()

gwo_weights = weights_to_simplex(alpha_pos)
print(f'GWO combination weights: RNN={gwo_weights[0]:.4f}, LSTM={gwo_weights[1]:.4f}, BPNN={gwo_weights[2]:.4f}')

In [ ]:
y_rnn = model_rnn.predict(X_test_s, verbose=0)
y_lstm = model_lstm.predict(X_test_s, verbose=0)
y_bpnn = model_bpnn.predict(X_test_s, verbose=0)
y_comb_s = gwo_weights[0]*y_rnn + gwo_weights[1]*y_lstm + gwo_weights[2]*y_bpnn
y_pred_combo = scaler_y.inverse_transform(y_comb_s.reshape(-1, 1)).reshape(y_test.shape)
y_pred_combo = np.clip(y_pred_combo, 0, 100).astype(np.float32)

def naive_predictor(X, horizon):
    last = X[:, -1, 0]
    return np.tile(last.reshape(-1, 1), (1, horizon))
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

mae_c = calculate_mae(y_test, y_pred_combo)
rmse_c = calculate_rmse(y_test, y_pred_combo)
mase_c = calculate_mase(y_test, y_pred_combo, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "SSA+RNN/LSTM/BPNN+GWO", "MAE": mae_c, "RMSE": rmse_c, "MASE": mase_c},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY (same format as notebook 10)")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(f"\n{results_df.to_string(index=False)}")
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: SSA + RNN/LSTM/BPNN + GWO', y=1.02, fontsize=12)
plt.show()

In [ ]:
naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive Baseline (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_combo[i], '-', linewidth=1.6, label='SSA+Combo+GWO')
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted (solid) vs Actual (dashed)')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_combo.mean(axis=0), '-', linewidth=1.6, label='SSA+Combo+GWO (mean)')
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mae_per_hour_c = np.abs(y_test - y_pred_combo).mean(axis=0)
mae_per_hour_naive = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour_c, '-o', label='SSA+Combo+GWO', markersize=4)
ax.plot(hours, mae_per_hour_naive, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
residuals_best = (y_test - y_pred_combo).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_best, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: SSA+Combo+GWO')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive Baseline')
plt.suptitle('Residual distribution (centered at 0 is ideal)', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): SSA+Combo+GWO = {residuals_best.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         SSA+Combo+GWO = {residuals_best.std():.4f}, Naive = {residuals_naive.std():.4f}')

### Key insights

- **Paper:** SSA decomposes and reconstructs (first 10 components); RNN, LSTM, BPNN each predict on reconstructed data; **GWO** optimizes the combination weights so that y = w_rnn·y_rnn + w_lstm·y_lstm + w_bpnn·y_bpnn minimizes validation error (paper reports RNN ~98.6%, BPNN ~0.005%, LSTM ~1.4%).
- **Improvement over naive:** Positive % means the combined model beats the last-value baseline.
- **Mean profile / per-hour MAE / residuals:** Same interpretation as notebooks 10–19.

In [ ]:
best_row = results_df[results_df['Model'] == 'SSA+RNN/LSTM/BPNN+GWO'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"Best model: SSA+RNN/LSTM/BPNN+GWO (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}).")
print(f"Improvement over Naive: MAE {imp_mae:+.1f}%.")